# Tile Generation Walkthrough

Step-by-step validation of the M4 tile-generation workflow using the pilot slide `CHN_AU_10_19-21`.
The notebook inspects the cut manifest and cut-local annotations, previews the multi-scale tile grids, runs tile generation for one cut, inspects sample tiles, and checks idempotent reuse of the written manifest.

## 1. Setup

This notebook expects the pilot cut TIFFs, `{stem}_cuts.json` manifest, and cut-local annotation GeoJSON files to exist under `data/cuts/CHN_AU_10_19-21/`.

In [ ]:
from __future__ import annotations

import json
import random
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import pandas as pd
import tifffile
from IPython.display import display
from PIL import Image

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data_preparation.generate_tiles import (
    DEFAULT_MIN_TISSUE_FRACTION,
    DEFAULT_OVERLAP,
    DEFAULT_TILE_SIZES,
    _compute_tile_grid,
    generate_tiles_batch,
    generate_tiles_for_cut,
)



In [ ]:
CUTS_DIR = REPO_ROOT / 'data' / 'cuts'
OUTPUT_DIR = REPO_ROOT / 'data' / 'tiles'
STEM = 'CHN_AU_10_19-21'
CUT_NAME = f'{STEM}_cut000'
CUT_TIFF_PATH = CUTS_DIR / STEM / f'{CUT_NAME}.tif'
ANNOTATIONS_PATH = CUTS_DIR / STEM / f'{CUT_NAME}_annotations.geojson'
CUTS_MANIFEST_PATH = CUTS_DIR / STEM / f'{STEM}_cuts.json'
PREVIEW_PATH = CUTS_DIR / STEM / f'{CUT_NAME}_preview.png'

assert CUT_TIFF_PATH.exists(), CUT_TIFF_PATH
assert ANNOTATIONS_PATH.exists(), ANNOTATIONS_PATH
assert CUTS_MANIFEST_PATH.exists(), CUTS_MANIFEST_PATH
assert PREVIEW_PATH.exists(), PREVIEW_PATH

## 2. Inspect inputs

Load the pilot stem's cuts manifest and the cut-local annotations for `cut000`.

In [ ]:
with CUTS_MANIFEST_PATH.open('r', encoding='utf-8') as fp:
    cuts_manifest = json.load(fp)
with ANNOTATIONS_PATH.open('r', encoding='utf-8') as fp:
    annotations_geojson = json.load(fp)

cuts_df = pd.DataFrame(
    [
        {
            'cut_index': cut['index'],
            'cut_name': cut['name'],
            'x0': cut['level0_bbox']['x0'],
            'y0': cut['level0_bbox']['y0'],
            'x1': cut['level0_bbox']['x1'],
            'y1': cut['level0_bbox']['y1'],
            'width': cut['level0_size'][0],
            'height': cut['level0_size'][1],
        }
        for cut in cuts_manifest['cuts']
    ]
)
display(cuts_df)

stage_counts = Counter(
    (feature.get('properties') or {}).get('classification', {}).get('name', 'Unknown')
    for feature in annotations_geojson.get('features', [])
)
summary_df = pd.DataFrame(
    [
        {'stage': stage, 'count': count}
        for stage, count in sorted(stage_counts.items())
    ]
)
display(summary_df)

## 3. Tile grid preview

Compute the tile grids for all four sizes and overlay them on the preview image.

In [ ]:
plt.figure(figsize=(12, 16))
plt.imshow(preview)
plt.title(f'{CUT_NAME} Preview (no tiling overlay)')
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
cut_entry = next(cut for cut in cuts_manifest['cuts'] if cut['name'] == CUT_NAME)
cut_width, cut_height = cut_entry['level0_size']
preview = Image.open(PREVIEW_PATH)
preview_width, preview_height = preview.size
scale_x = preview_width / cut_width
scale_y = preview_height / cut_height

grid_rows = []
grids_by_size = {}
for tile_size in DEFAULT_TILE_SIZES:
    stride = round(tile_size * (1 - DEFAULT_OVERLAP))
    grid = _compute_tile_grid(cut_h=cut_height, cut_w=cut_width, tile_size=tile_size, stride=stride)
    grids_by_size[tile_size] = grid
    grid_rows.append(
        {
            'tile_size': tile_size,
            'stride': stride,
            'n_rows': max(tile['row'] for tile in grid) + 1,
            'n_cols': max(tile['col'] for tile in grid) + 1,
            'total_tiles': len(grid),
        }
    )

display(pd.DataFrame(grid_rows))

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
for ax, tile_size in zip(axes.flat, DEFAULT_TILE_SIZES):
    ax.imshow(preview)
    ax.set_title(f'{tile_size}px tiles')
    ax.axis('off')
    for tile in grids_by_size[tile_size]:
        rect = patches.Rectangle(
            (tile['x0'] * scale_x, tile['y0'] * scale_y),
            (tile['x1'] - tile['x0']) * scale_x,
            (tile['y1'] - tile['y0']) * scale_y,
            linewidth=0.3,
            edgecolor='cyan',
            facecolor='none',
            alpha=0.35,
        )
        ax.add_patch(rect)
plt.tight_layout()

## 4. Generate tiles for cut000

Run the tile generator for one cut and assert that all referenced PNG files exist.

In [ ]:
manifest = generate_tiles_for_cut(
    CUT_TIFF_PATH,
    ANNOTATIONS_PATH,
    CUTS_MANIFEST_PATH,
    OUTPUT_DIR,
    skip_if_exists=False,
)

manifest_path = OUTPUT_DIR / STEM / CUT_NAME / f'{CUT_NAME}_tile_manifest.json'
assert manifest_path.exists()
assert manifest['n_tiles_total'] == len(manifest['tiles'])
assert manifest['n_tiles_with_oocyte'] > 0
assert set(manifest['tile_sizes']) == set(DEFAULT_TILE_SIZES)

grid_counts = {row['tile_size']: row['total_tiles'] for row in grid_rows}
retained_counts = Counter(tile['tile_size'] for tile in manifest['tiles'])
for tile in manifest['tiles']:
    png_path = REPO_ROOT / tile['png_path']
    assert png_path.exists(), png_path
    with Image.open(png_path) as image:
        assert image.size == (tile['tile_size'], tile['tile_size'])

display(pd.DataFrame(
    [
        {
            'tile_size': size,
            'grid_total': grid_counts[size],
            'written_tiles': retained_counts[size],
        }
        for size in DEFAULT_TILE_SIZES
    ]
))

## 5. Inspect sample tiles

Display four tiles with oocytes and four without for each tile size.

In [ ]:
rng = random.Random(0)
for tile_size in DEFAULT_TILE_SIZES:
    size_tiles = [tile for tile in manifest['tiles'] if tile['tile_size'] == tile_size]
    oocyte_tiles = [tile for tile in size_tiles if tile['has_oocyte']]
    background_tiles = [tile for tile in size_tiles if not tile['has_oocyte']]
    n_show = min(4, len(oocyte_tiles), len(background_tiles))
    if n_show == 0:
        print(f'No paired samples available for {tile_size}px tiles')
        continue

    sampled_oocyte = rng.sample(oocyte_tiles, n_show)
    sampled_background = rng.sample(background_tiles, n_show)
    fig, axes = plt.subplots(2, n_show, figsize=(4 * n_show, 8))
    for col, tile in enumerate(sampled_oocyte):
        image = Image.open(REPO_ROOT / tile['png_path'])
        axes[0, col].imshow(image)
        axes[0, col].set_title(f"{tile['tile_id']}\nTF={tile['tissue_fraction']:.2f}")
        axes[0, col].axis('off')
    for col, tile in enumerate(sampled_background):
        image = Image.open(REPO_ROOT / tile['png_path'])
        axes[1, col].imshow(image)
        axes[1, col].set_title(f"{tile['tile_id']}\nTF={tile['tissue_fraction']:.2f}")
        axes[1, col].axis('off')
    fig.suptitle(f'{tile_size}px sample tiles', fontsize=14)
    plt.tight_layout()
    plt.show()

## 6. Manifest statistics

Reload the written manifest from disk and inspect per-size and per-stage counts.

In [ ]:
with manifest_path.open('r', encoding='utf-8') as fp:
    manifest_on_disk = json.load(fp)

size_counts = Counter(tile['tile_size'] for tile in manifest_on_disk['tiles'])
stage_counts = Counter(stage for tile in manifest_on_disk['tiles'] for stage in tile['stages'])
stats_df = pd.DataFrame(
    [
        {'metric': 'n_tiles_total', 'value': manifest_on_disk['n_tiles_total']},
        {'metric': 'n_tiles_with_oocyte', 'value': manifest_on_disk['n_tiles_with_oocyte']},
        {'metric': 'n_tiles_skipped_tissue', 'value': manifest_on_disk['n_tiles_skipped_tissue']},
    ]
)
display(stats_df)
display(pd.DataFrame([{'tile_size': size, 'count': size_counts[size]} for size in DEFAULT_TILE_SIZES]))
display(pd.DataFrame([{'stage': stage, 'count': count} for stage, count in sorted(stage_counts.items())]))

assert manifest_on_disk['n_tiles_with_oocyte'] > 0
assert manifest_on_disk['n_tiles_total'] == sum(size_counts.values())

## 7. Batch run

Run the batch entry point with `skip_if_exists=True` and review a compact per-cut summary.

In [ ]:
batch_manifests = generate_tiles_batch(CUTS_DIR, OUTPUT_DIR, skip_if_exists=True, stem_glob=STEM)
batch_df = pd.DataFrame(
    [
        {
            'stem': item['stem'],
            'cut_index': item['cut_index'],
            'n_tiles_total': item['n_tiles_total'],
            'n_tiles_with_oocyte': item['n_tiles_with_oocyte'],
            'n_tiles_skipped_tissue': item['n_tiles_skipped_tissue'],
        }
        for item in batch_manifests
    ]
)
display(batch_df)
assert len(batch_manifests) >= 3

## 8. Skip-if-exists idempotency

Confirm that rerunning `generate_tiles_for_cut` with `skip_if_exists=True` returns the cached manifest without rewriting it.

In [ ]:
before_mtime = manifest_path.stat().st_mtime_ns
cached_manifest = generate_tiles_for_cut(
    CUT_TIFF_PATH,
    ANNOTATIONS_PATH,
    CUTS_MANIFEST_PATH,
    OUTPUT_DIR,
    skip_if_exists=True,
)
after_mtime = manifest_path.stat().st_mtime_ns

assert cached_manifest['n_tiles_total'] == manifest_on_disk['n_tiles_total']
assert after_mtime == before_mtime
print('skip_if_exists returned cached manifest without rewriting it')

## 9. Review analysis

Load the completed collaborator review session and compute report-only agreement metrics by tile size. This section documents the returned labels without changing the review protocol or dropping any tile scales.


In [ ]:
REVIEW_SESSION_PATH = REPO_ROOT / 'data' / 'review_sessions' / '2026-05-27_review001' / 'session.json'

assert REVIEW_SESSION_PATH.exists(), REVIEW_SESSION_PATH
with REVIEW_SESSION_PATH.open('r', encoding='utf-8') as fp:
    review_session = json.load(fp)

review_tiles_df = pd.DataFrame(
    [
        {
            'display_index': tile['display_index'],
            'tile_size': tile['tile_size'],
            'ground_truth': tile['ground_truth'],
            'collaborator_label': tile['collaborator_label'],
            'correct': tile['ground_truth'] == tile['collaborator_label'],
            'cut_name': tile['cut_name'],
            'tile_id': tile['tile_id'],
            'png_filename': tile['png_filename'],
        }
        for tile in review_session['tiles']
    ]
).sort_values('display_index')

n_tiles = len(review_tiles_df)
n_labelled = int(review_tiles_df['collaborator_label'].notna().sum())
n_positive = int(review_tiles_df['ground_truth'].sum())
n_negative = int((~review_tiles_df['ground_truth'].astype(bool)).sum())
size_counts = review_tiles_df['tile_size'].value_counts().sort_index()

summary_df = pd.DataFrame(
    [
        {'metric': 'session_id', 'value': review_session['session_id']},
        {'metric': 'n_tiles', 'value': n_tiles},
        {'metric': 'n_labelled', 'value': n_labelled},
        {'metric': 'n_ground_truth_positive', 'value': n_positive},
        {'metric': 'n_ground_truth_negative', 'value': n_negative},
        {'metric': 'overall_accuracy', 'value': review_tiles_df['correct'].mean()},
    ]
)

display(summary_df)
display(size_counts.rename_axis('tile_size').reset_index(name='count'))

assert n_tiles == 40
assert n_labelled == 40
assert n_positive == 20
assert n_negative == 20
assert size_counts.to_dict() == {128: 10, 256: 10, 512: 10, 1024: 10}


In [ ]:
def _review_metrics(group: pd.DataFrame) -> pd.Series:
    ground_truth = group['ground_truth'].astype(bool)
    predicted = group['collaborator_label'].astype(bool)
    tp = int((ground_truth & predicted).sum())
    tn = int((~ground_truth & ~predicted).sum())
    fp = int((~ground_truth & predicted).sum())
    fn = int((ground_truth & ~predicted).sum())
    total = int(len(group))
    sensitivity_denominator = tp + fn
    specificity_denominator = tn + fp
    return pd.Series(
        {
            'total': total,
            'tp': tp,
            'tn': tn,
            'fp': fp,
            'fn': fn,
            'accuracy': (tp + tn) / total,
            'sensitivity': tp / sensitivity_denominator if sensitivity_denominator else pd.NA,
            'specificity': tn / specificity_denominator if specificity_denominator else pd.NA,
        }
    )

metrics_df = (
    review_tiles_df.groupby('tile_size', sort=True)
    .apply(_review_metrics, include_groups=False)
    .reset_index()
)

expected_metrics_df = pd.DataFrame(
    [
        {'tile_size': 128, 'accuracy': 0.50, 'sensitivity': 0.00, 'specificity': 1.00},
        {'tile_size': 256, 'accuracy': 0.50, 'sensitivity': 0.00, 'specificity': 1.00},
        {'tile_size': 512, 'accuracy': 0.70, 'sensitivity': 0.60, 'specificity': 0.80},
        {'tile_size': 1024, 'accuracy': 0.90, 'sensitivity': 1.00, 'specificity': 0.80},
    ]
)

metrics_display_df = metrics_df.copy()
for column in ['accuracy', 'sensitivity', 'specificity']:
    metrics_display_df[column] = metrics_display_df[column].astype(float).round(2)

display(metrics_display_df)

pd.testing.assert_frame_equal(
    metrics_display_df[['tile_size', 'accuracy', 'sensitivity', 'specificity']].reset_index(drop=True),
    expected_metrics_df,
    check_dtype=False,
)


In [ ]:
confusion_df = (
    review_tiles_df.groupby(['ground_truth', 'collaborator_label'], dropna=False)
    .size()
    .reset_index(name='count')
)
mismatches_df = review_tiles_df.loc[
    ~review_tiles_df['correct'],
    [
        'display_index',
        'tile_size',
        'ground_truth',
        'collaborator_label',
        'cut_name',
        'tile_id',
        'png_filename',
    ],
].reset_index(drop=True)

print(f"Overall accuracy: {review_tiles_df['correct'].sum()} / {n_tiles} = {review_tiles_df['correct'].mean():.2f}")
display(confusion_df)
display(mismatches_df)

assert len(mismatches_df) == 14
